<a href="https://colab.research.google.com/github/akashmavle5/--akash/blob/main/Akash_Investment_algo001.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# graham_50_scrape.py
import requests
from bs4 import BeautifulSoup
import yfinance as yf
import pandas as pd
import time
from tqdm import tqdm

# ---------- USER: insert your list of tickers (NSE tickers) ----------
tickers = [
"RELIANCE","HDFCBANK","INFY","TCS","ICICIBANK","KOTAKBANK","SBIN","HINDUNILVR","NESTLEIND","BHARTIARTL",
"BAJFINANCE","BAJAJFINSV","LT","M&M","MARUTI","BAJAJ-AUTO","ASIANPAINT","TITAN","ULTRATECH","NTPC",
"POWERGRID","COALINDIA","ONGC","BPCL","HINDALCO","HINDZINC","ADANIPORTS","ADANIENT","APOLLOHOSP","DRREDDY",
"SUNPHARMA","CIPLA","DIVISLAB","TRENT","DIXON","CHOLAFIN","HDFCLIFE","BEL","PFC","RECLTD",
"CHAMBLFERT","ASHOKA","TANLA","NUCLEUS","AJANTPHARM","TECHM","BRITANNIA","PIDILITIND","HAVELLS","PBFINTECH"
]

# Helper: Screener.in consolidated page (some tickers may have slight naming differences on Screener.in)
def screener_url(ticker):
    # Screener uses certain company slugs, many are the same as ticker but some differ.
    # We try ticker first; if 404, fall back to a simple mapping (expand mapping if you find 404s)
    base = "https://www.screener.in/company/{}/consolidated/".format(ticker)
    return base

def parse_screener_metrics(ticker):
    url = screener_url(ticker)
    headers = {"User-Agent":"Mozilla/5.0"}
    out = {
        "ROCE": None,
        "ROE": None,
        "Sales_3Y_Growth%": None,
        "Profit_3Y_Growth%": None,
        "P/E_screener": None,
        "Debt_to_Equity_screener": None
    }
    try:
        r = requests.get(url, headers=headers, timeout=12)
        if r.status_code != 200:
            return out
        soup = BeautifulSoup(r.text, "lxml")
        # Screener shows key ratios in li.flex.flex-space-between
        for li in soup.select("li.flex.flex-space-between"):
            try:
                name_tag = li.find("span", class_="name")
                value_tag = li.find("span", class_="value")
                if not name_tag or not value_tag:
                    continue
                name = name_tag.get_text(strip=True)
                value = value_tag.get_text(strip=True).replace(',', '').replace('%','').strip()
                # map the common names
                if "Return on capital employed" in name or name=="ROCE":
                    out["ROCE"] = float(value) if value not in ["-", ""] else None
                if "Return on equity" in name or name=="ROE":
                    out["ROE"] = float(value) if value not in ["-", ""] else None
                if "Sales growth 3Years" in name or "Sales growth (3Y)" in name:
                    out["Sales_3Y_Growth%"] = float(value) if value not in ["-", ""] else None
                if "Profit growth 3Years" in name or "Profit growth (3Y)" in name:
                    out["Profit_3Y_Growth%"] = float(value) if value not in ["-", ""] else None
                if "P/E" in name and out["P/E_screener"] is None:
                    out["P/E_screener"] = float(value) if value not in ["-", ""] else None
                if "Debt to equity" in name:
                    out["Debt_to_Equity_screener"] = float(value) if value not in ["-", ""] else None
            except Exception:
                continue
    except Exception:
        pass
    return out

def fetch_yahoo(ticker):
    # append .NS for NSE tickers to use yfinance
    t = ticker + ".NS"
    try:
        y = yf.Ticker(t)
        info = y.info
        return {
            "market_cap": info.get("marketCap"),
            "trailingPE": info.get("trailingPE"),
            "debtToEquity": info.get("debtToEquity"),
            "freeCashflow": info.get("freeCashflow"),
            "price": info.get("regularMarketPrice"),
            "beta": info.get("beta"),
            "priceToBook": info.get("priceToBook"),
            "returnOnEquity": info.get("returnOnEquity"), # decimal
            "revenueGrowth": info.get("revenueGrowth"),   # decimal
            "ebitda": info.get("ebitda"),
        }
    except Exception:
        return {}

# Scoring function (simple, adjustable)
def graham_score(row):
    score = 0
    # P/E rule: lower better (<=25 gets points)
    pe = row.get("trailingPE")
    if pe and pe>0:
        if pe <= 12:
            score += 25
        elif pe <= 20:
            score += 15
        elif pe <= 25:
            score += 8
    # ROCE
    roce = row.get("ROCE")
    if roce:
        if roce >= 30: score += 20
        elif roce >= 20: score += 12
        elif roce >= 15: score += 6
    # ROE
    roe = row.get("ROE") or (row.get("returnOnEquity") and row.get("returnOnEquity")*100)
    if roe:
        if roe >= 30: score += 10
        elif roe >= 20: score += 6
        elif roe >= 15: score += 3
    # Revenue growth (3Y)
    s3 = row.get("Sales_3Y_Growth%")
    if s3:
        if s3 >= 20: score += 8
        elif s3 >= 10: score += 4
    # Profit growth (3Y)
    p3 = row.get("Profit_3Y_Growth%")
    if p3:
        if p3 >= 20: score += 8
        elif p3 >= 10: score += 4
    # Debt
    dte = row.get("debtToEquity")
    if dte is not None:
        if dte <= 0.5: score += 6
        elif dte <= 1: score += 3
    # Free cash flow positive
    if row.get("freeCashflow") and row.get("freeCashflow") > 0:
        score += 6
    return score

# Main loop
rows = []
for tk in tqdm(tickers):
    sc = parse_screener_metrics(tk)
    y = fetch_yahoo(tk)
    time.sleep(1)  # polite delay
    row = {
        "Ticker": tk,
        "Market_Cap": y.get("market_cap"),
        "Price": y.get("price"),
        "trailingPE": y.get("trailingPE") or sc.get("P/E_screener"),
        "ROCE": sc.get("ROCE"),
        "ROE": sc.get("ROE") or (y.get("returnOnEquity") and y.get("returnOnEquity")*100),
        "Sales_3Y_Growth%": sc.get("Sales_3Y_Growth%"),
        "Profit_3Y_Growth%": sc.get("Profit_3Y_Growth%"),
        "Debt_to_Equity": y.get("debtToEquity") if y.get("debtToEquity") is not None else sc.get("Debt_to_Equity_screener"),
        "Free_Cashflow": y.get("freeCashflow"),
        "PriceToBook": y.get("priceToBook"),
        "Beta": y.get("beta"),
        "EBITDA": y.get("ebitda")
    }
    row["Graham_Score"] = graham_score(row)
    rows.append(row)

df = pd.DataFrame(rows)
# reorder and format columns
cols = ["Ticker","Market_Cap","Price","trailingPE","ROCE","ROE","Sales_3Y_Growth%","Profit_3Y_Growth%","Debt_to_Equity","Free_Cashflow","PriceToBook","EBITDA","Graham_Score"]
df = df[cols]
df = df.sort_values(by="Graham_Score", ascending=False)

df.to_csv("graham_50_results.csv", index=False)
df.to_excel("graham_50_results.xlsx", index=False)
print("Saved graham_50_results.csv and graham_50_results.xlsx")
print(df.head(50).to_string(index=False))


100%|██████████| 50/50 [01:46<00:00,  2.12s/it]


Saved graham_50_results.csv and graham_50_results.xlsx
    Ticker   Market_Cap    Price  trailingPE  ROCE   ROE Sales_3Y_Growth% Profit_3Y_Growth%  Debt_to_Equity  Free_Cashflow  PriceToBook       EBITDA  Graham_Score
    ASHOKA 5.920201e+10   210.85    3.361231 39.70 54.80             None              None          47.793            NaN     1.511491 2.957432e+10            55
 COALINDIA 2.389906e+12   387.80    7.657978 48.00 38.90             None              None          12.976   1.264044e+11     2.266815 4.364282e+11            55
  HINDZINC 2.031956e+12   480.90   19.220623 61.00 72.90             None              None          88.173   5.980875e+10    14.833437 1.773300e+11            45
CHAMBLFERT 1.956592e+11   488.25   11.183005 26.80 19.80             None              None           1.134            NaN     2.243817 2.495973e+10            40
       TCS 1.090528e+13  3014.10   22.050627 64.60 52.40             None              None          10.173   3.926525e+11    10.2